In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import dotenv

#自定义中间件wrap-style hooks
load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek"
)

In [3]:

from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware


@wrap_model_call
def warp_model_call_middleware(request:ModelRequest,handler:Callable[[ModelRequest],ModelResponse])->ModelResponse|None:
    request.messages[-1].content+="-----> wrap_model_call_before<------"
    #模型调用
    response=handler(request)
    response.result[0].content+="-----> wrap_model_call_after<------"
    return response
agent=create_agent(
    model=model,
    middleware=[warp_model_call_middleware]
)
response=agent.invoke({
    "messages":[HumanMessage("你好")]
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好-----> wrap_model_call_before<------
================================== Ai Message ==================================

你好！我注意到你发来了“wrap_model_call_before”，但这个含义我不太确定。你是想咨询 API 调用相关的问题吗？还是需要我帮忙处理什么任务？请告诉我你的具体需求，我会尽力协助你！-----> wrap_model_call_after<------


In [5]:
# 基于类的实现
from langchain.agents.middleware import AgentMiddleware
class WrapModelCallMiddleware(AgentMiddleware):
   def warp_model_call(self,request:ModelRequest,handler:Callable[[ModelRequest],ModelResponse])->ModelResponse|None:
    request.messages[-1].content+="-----> wrap_model_call_before<------"
    #模型调用
    response=handler(request)
    response.result[0].content+="-----> wrap_model_call_after<------"
    return response 
agent=create_agent(
    model=model,
    middleware=[WrapModelCallMiddleware()]
)
response=agent.invoke({
    "messages":[HumanMessage("你好")]
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！很高兴见到你。有什么我可以帮你的吗？


In [6]:
from langchain_core.tools import tool
from langgraph.prebuilt.tool_node import ToolCallRequest
from typing import Any
from langchain_core.messages import ToolMessage
from langgraph.types import Command
from langchain.agents.middleware import wrap_tool_call
@tool
def get_weather(city:str,is_forcast:bool) -> str:
    """
    获取当日特定城市的天气
    
    Args:
        city:城市名称
        is_forcast:是否包含明天天气预报
    """
    res=f"{city}今天天气不错"
    if is_forcast:
        res +="\n明天天气也很好"
    return res


#2. wrap_tool_call
@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
        handler: Callable[[ToolCallRequest],ToolMessage|Command[Any]])->ToolMessage|Command[Any]:
    result=handler(request)
    print(f"原始的参数：{request.tool_call['args']}")
    print(f"原始参数调用的结果：{result}")
    request.tool_call["args"]["is_forcast"]=True
    result=handler(request)
    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用的结果：{result}")
    return result
agent=create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)
response=agent.invoke({
    "messages":[HumanMessage("帮我查询北京今天的天气如何！")]
})
for msg in response["messages"]:
    msg.pretty_print() 

原始的参数：{'city': '北京', 'is_forcast': False}
原始参数调用的结果：content='北京今天天气不错' name='get_weather' tool_call_id='call_00_Trq5k2oUuGqBj4poOafQ3639'
更新以后的参数：{'city': '北京', 'is_forcast': True}
更新以后的参数调用的结果：content='北京今天天气不错\n明天天气也很好' name='get_weather' tool_call_id='call_00_Trq5k2oUuGqBj4poOafQ3639'
================================ Human Message =================================

帮我查询北京今天的天气如何！
================================== Ai Message ==================================

我来帮您查询北京今天的天气情况。
Tool Calls:
  get_weather (call_00_Trq5k2oUuGqBj4poOafQ3639)
 Call ID: call_00_Trq5k2oUuGqBj4poOafQ3639
  Args:
    city: 北京
    is_forcast: True
================================= Tool Message =================================
Name: get_weather

北京今天天气不错
明天天气也很好
================================== Ai Message ==================================

已为您查询到北京今天的天气情况！☀️

**北京今天的天气**：天气不错

另外还为您获取了明天的天气预报：明天天气也很好。

整体来看，北京最近两天的天气状况都比较理想，适合外出活动。如果您需要更详细的信息（如具体温度、降雨概率等），请随时告诉我！
